In [2]:
import numpy as np
import pandas as pd

BUDGET = 1_000_000


def optimize_manual_portfolio(expected_returns, max_total_volume_pct=100):
    """
    expected_returns:
        dict of product -> expected return

        Positive return means long.
        Negative return means short.

        Example:
        {
            "SULFUR_LTD": 0.35,
            "LAVA_CAKES": -0.30,
        }
    """

    rows = []

    for product, r in expected_returns.items():
        if r == 0 or pd.isna(r):
            continue

        rows.append(
            {
                "product": product,
                "side": "BUY" if r > 0 else "SELL",
                "expected_return": r,
                "edge": abs(r),
            }
        )

    df = pd.DataFrame(rows)

    if df.empty:
        return df, {}

    df = df.sort_values("edge", ascending=False).reset_index(drop=True)

    edges = df["edge"].values
    max_total_w = max_total_volume_pct / 100

    # Unconstrained optimum: w_i = edge_i / 2
    w_unconstrained = edges / 2

    if w_unconstrained.sum() <= max_total_w:
        w = w_unconstrained
    else:
        # Budget-constrained case:
        # w_i = max(0, (edge_i - lambda) / 2)
        lo, hi = 0, edges.max()

        for _ in range(100):
            lam = (lo + hi) / 2
            w_candidate = np.maximum(0, (edges - lam) / 2)

            if w_candidate.sum() > max_total_w:
                lo = lam
            else:
                hi = lam

        w = np.maximum(0, (edges - hi) / 2)

    df["volume_pct"] = 100 * w
    df["fee"] = BUDGET * w**2
    df["gross_expected_pnl"] = BUDGET * w * df["edge"]
    df["net_expected_pnl"] = df["gross_expected_pnl"] - df["fee"]

    df = df[df["volume_pct"] > 1e-9].copy()

    summary = {
        "total_volume_pct": df["volume_pct"].sum(),
        "total_fee": df["fee"].sum(),
        "total_gross_expected_pnl": df["gross_expected_pnl"].sum(),
        "total_net_expected_pnl": df["net_expected_pnl"].sum(),
    }

    return df, summary


expected_returns = {
    "SULFUR_LTD": 0.35,
    "LAVA_CAKES": -0.30,
    "THERMALITE_CORES": 0.25,
    "PYROFLEX_CELL": -0.20,
    "MAGMA_INK": 0.18,
    "SCORIA_PASTE": 0.15,
    "ASHES_OF_THE_PHOENIX": -0.12,
    "VOLCANIC_INCENSE": 0.10,
    "OBSIDIAN_CUTLERY": 0.08,
}

portfolio, summary = optimize_manual_portfolio(expected_returns)

display(portfolio)
print(summary)

,product,side,expected_return,edge,volume_pct,fee,gross_expected_pnl,net_expected_pnl
0,SULFUR_LTD,BUY,0.35,0.35,17.5,30625.0,61250.0,30625.0
1,LAVA_CAKES,SELL,-0.30,0.30,15.0,22500.0,45000.0,22500.0
2,THERMALITE_CORES,BUY,0.25,0.25,12.5,15625.0,31250.0,15625.0
3,PYROFLEX_CELL,SELL,-0.20,0.20,10.0,10000.0,20000.0,10000.0
4,MAGMA_INK,BUY,0.18,0.18,9.0,8100.0,16200.0,8100.0
5,SCORIA_PASTE,BUY,0.15,0.15,7.5,5625.0,11250.0,5625.0
6,ASHES_OF_THE_PHOENIX,SELL,-0.12,0.12,6.0,3600.0,7200.0,3600.0
7,VOLCANIC_INCENSE,BUY,0.10,0.10,5.0,2500.0,5000.0,2500.0
8,OBSIDIAN_CUTLERY,BUY,0.08,0.08,4.0,1600.0,3200.0,1600.0


{'total_volume_pct': np.float64(86.5), 'total_fee': np.float64(100175.0), 'total_gross_expected_pnl': np.float64(200350.0), 'total_net_expected_pnl': np.float64(100175.0)}
